In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM mbtae10", con=connection2)

connection2.close()



In [2]:
base2

,tipoateemecod,tipoateemedes
0,1,ATENCION EMERGENCIA
1,2,INTERCONSULTA


In [3]:
a=base2[base2.duplicated(subset=["tipoateemecod"])]
a

,tipoateemecod,tipoateemedes


In [4]:
base2.columns

Index(['tipoateemecod', 'tipoateemedes'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbtae10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbtae10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbtae10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbtae10 
ALTER COLUMN tipoateemecod TYPE CHARACTER (1),
ALTER COLUMN tipoateemedes TYPE CHARACTER (20);


UPDATE mbtae10 
SET 

tipoateemedes= t.tipoateemedes


FROM tmp_mbtae10 t 
WHERE mbtae10.tipoateemecod = t.tipoateemecod and mbtae10.tipoateemecod IS NOT NULL and t.tipoateemecod IS NOT NULL ;

INSERT INTO mbtae10 
(tipoateemecod, tipoateemedes) 

SELECT 
tipoateemecod, tipoateemedes

FROM tmp_mbtae10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbtae10 
    WHERE mbtae10.tipoateemecod = tmp_mbtae10.tipoateemecod and mbtae10.tipoateemecod IS NOT NULL and tmp_mbtae10.tipoateemecod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbtae10;
DELETE FROM mbtae10 WHERE tipoateemecod ='';
SELECT COUNT(*) FROM mbtae10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbtae10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbtae10 antes de la inserción: 2
Cantidad de filas en la tabla mbtae10 después de la inserción: 2
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['tipoateemecod', 'tipoateemedes'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbtae10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_emetipate"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emetipate antes de la inserción: {cant_antes}")


query="""
ALTER TABLE tmp_mbtae10 
ALTER COLUMN tipoateemecod TYPE CHARACTER (1),
ALTER COLUMN tipoateemedes TYPE CHARACTER (20);



INSERT INTO dim_emetipate (cod_tip, des_tip) 
SELECT tipoateemecod, tipoateemedes

FROM tmp_mbtae10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emetipate 
    WHERE dim_emetipate.cod_tip = tmp_mbtae10.tipoateemecod
);

DROP TABLE tmp_mbtae10;

SELECT COUNT(*) FROM dim_emetipate;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emetipate después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_emetipate antes de la inserción: 2
Cantidad de filas en la tabla dim_emetipate después de la inserción: 2
La cantidad de filas insertadas fue de: 0
